In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

train_df = pd.read_csv('../data/processed/train_clean.csv')
test_df_full = pd.read_csv('../data/processed/test_clean.csv')

# Separar PassengerId (que ahora viene incluido en el archivo) del resto de columnas
passenger_ids_test = test_df_full['PassengerId']
test_df = test_df_full.drop(columns=['PassengerId'])

train_df.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,HasCabin,FamilySize,IsAlone,Embarked_Q,...,Ticket_item_SOTONOQ,Ticket_item_STONO_2,Ticket_item_WC,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_Unknown
0,0,3,0,1,0,7.2500,0,2,0,False,...,False,False,False,False,False,False,False,False,False,True
1,1,1,1,1,0,71.2833,1,2,0,False,...,False,False,False,False,True,False,False,False,False,False
2,1,3,1,0,0,7.9250,0,1,1,False,...,False,False,False,False,False,False,False,False,False,True
3,1,1,1,1,0,53.1000,1,2,0,False,...,False,False,False,False,True,False,False,False,False,False
4,0,3,0,0,0,8.0500,0,1,1,False,...,False,False,False,False,False,False,False,False,False,True


In [20]:
X = train_df.drop(columns=['Survived'])
y = train_df['Survived']

print(f"Variables predictoras: {list(X.columns)}")
print(f"Forma de X: {X.shape}")

Variables predictoras: ['Pclass', 'Sex', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'FamilySize', 'IsAlone', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'AgeBin_Adolescente', 'AgeBin_Adulto', 'AgeBin_Adulto_mayor', 'AgeBin_Anciano', 'FareBin_Media', 'FareBin_Alta', 'FareBin_MuyAlta', 'Ticket_item_CA', 'Ticket_item_NONE', 'Ticket_item_PC', 'Ticket_item_RARE', 'Ticket_item_SCPARIS', 'Ticket_item_SOTONOQ', 'Ticket_item_STONO_2', 'Ticket_item_WC', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_Unknown']
Forma de X: (891, 36)


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} pasajeros")
print(f"Prueba: {X_test.shape[0]} pasajeros")

Entrenamiento: 712 pasajeros
Prueba: 179 pasajeros


In [22]:
modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.3f}")
print(f"F1-score: {f1_score(y_test, y_pred_lr):.3f}")

=== Logistic Regression ===
Accuracy: 0.821
Precision: 0.768
Recall: 0.768
F1-score: 0.768


In [23]:
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.3f}")
print(f"F1-score: {f1_score(y_test, y_pred_rf):.3f}")

=== Random Forest ===
Accuracy: 0.765
Precision: 0.696
Recall: 0.696
F1-score: 0.696


In [24]:
train_acc_lr = accuracy_score(y_train, modelo_lr.predict(X_train))
test_acc_lr = accuracy_score(y_test, modelo_lr.predict(X_test))

print("=== Logistic Regression - Train vs Test ===")
print(f"Accuracy en TRAIN: {train_acc_lr:.3f}")
print(f"Accuracy en TEST:  {test_acc_lr:.3f}")
print(f"Diferencia: {train_acc_lr - test_acc_lr:.3f}")

=== Logistic Regression - Train vs Test ===
Accuracy en TRAIN: 0.836
Accuracy en TEST:  0.821
Diferencia: 0.014


In [25]:
train_acc_rf = accuracy_score(y_train, modelo_rf.predict(X_train))
test_acc_rf = accuracy_score(y_test, modelo_rf.predict(X_test))

print("=== Random Forest - Train vs Test ===")
print(f"Accuracy en TRAIN: {train_acc_rf:.3f}")
print(f"Accuracy en TEST:  {test_acc_rf:.3f}")
print(f"Diferencia: {train_acc_rf - test_acc_rf:.3f}")

=== Random Forest - Train vs Test ===
Accuracy en TRAIN: 0.962
Accuracy en TEST:  0.765
Diferencia: 0.197


In [26]:
# random forest con balanceo de clases y búsqueda de hiperparámetros
from sklearn.model_selection import GridSearchCV

param_grid_balanced = {
    'n_estimators': [100, 200, 300],
    'max_depth': [6, 8, 10, 12, None],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [2, 4, 6],
    'max_features': ['sqrt', 'log2']
}

grid_search_balanced = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid_balanced,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_balanced.fit(X_train, y_train)

print(f"Mejores parámetros (balanced): {grid_search_balanced.best_params_}")
print(f"Mejor accuracy (cross-val) (balanced): {grid_search_balanced.best_score_:.3f}")

mejor_rf_balanced = grid_search_balanced.best_estimator_

Mejores parámetros (balanced): {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Mejor accuracy (cross-val) (balanced): 0.831


In [27]:
y_pred_mejor_rf_balanced_v2 = mejor_rf_balanced.predict(X_test)

print("=== Random Forest Optimizado (balanced) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mejor_rf_balanced_v2):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_mejor_rf_balanced_v2):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_mejor_rf_balanced_v2):.3f}")
print(f"F1-score: {f1_score(y_test, y_pred_mejor_rf_balanced_v2):.3f}")

=== Random Forest Optimizado (balanced) ===
Accuracy: 0.777
Precision: 0.679
Recall: 0.797
F1-score: 0.733


In [28]:
predicciones_rf_balanced_v2 = mejor_rf_balanced.predict(test_df)

submission_rf_balanced_v2 = pd.DataFrame({
    'PassengerId': passenger_ids_test,
    'Survived': predicciones_rf_balanced_v2.astype(int)
})

submission_rf_balanced_v2.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [29]:
train_acc_rf_balanced = accuracy_score(y_train, mejor_rf_balanced.predict(X_train))
test_acc_rf_balanced = accuracy_score(y_test, mejor_rf_balanced.predict(X_test))

print("=== Random Forest Optimizado (balanced) - Train vs Test ===")
print(f"Accuracy en TRAIN: {train_acc_rf_balanced:.3f}")
print(f"Accuracy en TEST:  {test_acc_rf_balanced:.3f}")
print(f"Diferencia: {train_acc_rf_balanced - test_acc_rf_balanced:.3f}")

=== Random Forest Optimizado (balanced) - Train vs Test ===
Accuracy en TRAIN: 0.857
Accuracy en TEST:  0.777
Diferencia: 0.080


In [30]:

predicciones_lr_v2 = modelo_lr.predict(test_df)

submission_lr_v2 = pd.DataFrame({
    'PassengerId': passenger_ids_test,
    'Survived': predicciones_lr_v2.astype(int)
})

submission_lr_v2.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [31]:
submission_lr_v2.to_csv('../outputs/prediccion_lr_v2.csv', index=False)
print("Archivo de predicción (Logistic Regression v2) guardado correctamente ✅")

Archivo de predicción (Logistic Regression v2) guardado correctamente ✅


In [32]:
predicciones_rf_balanced_v2 = mejor_rf_balanced.predict(test_df)

submission_rf_balanced_v2 = pd.DataFrame({
    'PassengerId': passenger_ids_test,
    'Survived': predicciones_rf_balanced_v2.astype(int)
})

submission_rf_balanced_v2.to_csv('../outputs/prediccion_rf_balanced_v2.csv', index=False)
print("Archivo de predicción (Random Forest balanced v2) guardado correctamente ✅")

Archivo de predicción (Random Forest balanced v2) guardado correctamente ✅
